# f4_m05 — Análisis Bivariante vs Target (`abandono`)


**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |

---

---

## Qué hace
Estudia la relación individual de cada *feature* con la variable objetivo `abandono` (0 = no abandona, 1 = abandona).  
Para variables **numéricas**: distribuciones por clase, Mann-Whitney U, Cohen's *d* con IC 95 % bootstrap.  
Para variables **categóricas**: tasas de abandono por categoría, Chi² de Pearson, Cramér's *V*.  
Se genera un **ranking global de asociación** y un deep-dive de las variables más relevantes.

## Requisitos
- `data/04_eda/df_eda_final.parquet` — dataset EDA con texto legible (mismo que M01-M04)
- Entorno activo con: `pandas`, `numpy`, `scipy`, `statsmodels`, `pingouin`, `seaborn`, `plotly`, `plotnine`, `altair`, `jinja2`
- Utilidades del proyecto: `formato_numero_es`, `render_base_html`

## Genera
- Visualizaciones inline (Seaborn, Plotly, Plotnine, Altair)
- `resultados_bivariante_numericas.csv` en `data/03_features/`
- `resultados_bivariante_categoricas.csv` en `data/03_features/`
- `docs/html/fase4/f4_m05_bivariante.html`

## Flujo
`M04 Categóricas` → **M05 Bivariante** → `M06 Correlaciones`

## Siguiente
`f4_m06_correlaciones.ipynb` — matriz de correlaciones y multicolinealidad entre features

In [1]:
# ── 0.1 Detección robusta de ROOT (busca src/ ascendiendo niveles) ──────────
import sys
from pathlib import Path

ROOT = Path.cwd()
for _ in range(5):
    if (ROOT / "src").exists(): break
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))
print(f"ROOT detectado: {ROOT}")


ROOT detectado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI


In [2]:
# ── CSS + JS para botones Interpretar ────────────────────────────────────────
js_css_html = (
    '<style>'
    '.ibt{display:inline-flex;align-items:center;gap:5px;padding:5px 12px;'
    'background:#3182ce;color:#fff;border:none;border-radius:5px;'
    'font-size:12px;font-weight:600;cursor:pointer;float:right;margin-top:-2px;}'
    '.ibt:hover{background:#2b6cb0;}'
    '.ipn{display:none;background:#f7fafc;border:1px solid #e2e8f0;'
    'border-radius:6px;padding:14px 16px;margin-top:8px;font-size:13px;'
    'color:#2d3748;line-height:1.6;}'
    '.ipn.vis{display:block;}'
    '.tg{background:#c6f6d5;color:#276749;padding:1px 6px;border-radius:3px;font-weight:600;}'
    '.tb{background:#fed7d7;color:#9b2c2c;padding:1px 6px;border-radius:3px;font-weight:600;}'
    '.tm{background:#fefcbf;color:#744210;padding:1px 6px;border-radius:3px;font-weight:600;}'
    '</style>'
    '<script>'
    'function tg(id){'
    'var p=document.getElementById(id);'
    'var b=document.querySelector("[data-pid=\'" + id + "\']");'
    'if(!p||!b)return;'
    'var vis=p.classList.toggle("vis");'
    'b.textContent=vis?"✖ Cerrar":"📖 Interpretar";}'
    '</script>'
)

def h2btn(titulo, pid):
    return (
        f'<div style="display:flex;align-items:center;justify-content:space-between;'
        f'margin:28px 0 8px;">'
        f'<h2 style="font-size:18px;font-weight:700;color:#1a202c;margin:0;">{titulo}</h2>'
        f'<button class="ibt" data-pid="{pid}" onclick="tg(\'{pid}\')">' 
        f'📖 Interpretar</button></div>'
    )

def panel(pid, texto):
    return f'<div id="{pid}" class="ipn">{texto}</div>'


In [3]:
# ── 0.2 Imports generales ────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
import pingouin as pg

import matplotlib
matplotlib.use("Agg")  # Backend no interactivo — necesario para base64
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from plotnine import (
    ggplot, aes, geom_density, geom_histogram,
    facet_wrap, scale_fill_manual, scale_color_manual,
    theme_minimal, theme, element_text, labs, coord_flip
)
import altair as alt

from IPython.display import display, HTML

# Utilidades del proyecto
import base64
import io

from src.config import (RUTA_HTML, info_entorno)
from src.utils import crear_directorios, formato_numero_es

# ── Paneles de interpretación dinámica ──────────────
panel_violin_num = panel('violin-num', 'Menor solapamiento entre grupos = mejor predictor. Coincide con ranking d Cohen M08: <span class=\"tg\">cred_superados_anio_1er</span> y <span class=\"tg\">n_anios_beca</span> dominan. Mann-Whitney confirma significación estadística.')
panel_barras_cat = panel('barras-cat', 'Una categoría con >40% abandono es señal de riesgo fuerte. Estas proporciones son la base de los perfiles del M08 y de las reglas de intervención preventiva.')
panel_mann_whitney = panel('mann-whitney', 'Test no paramétrico para comparar medianas. Con 33.621 estudiantes casi todo sale significativo (p<0.05). <span class=\"tm\">El d de Cohen es más informativo</span> que el p-value porque mide la magnitud del efecto, no solo si existe.')

from src.html import (generar_kpis_html, generar_seccion_html,
                      generar_html_navegacion_completa, guardar_html)
from src.html.render import render_base_html

RUTA_FASE4_HTML = RUTA_HTML / "fase4"
crear_directorios([RUTA_FASE4_HTML])

print("Imports OK")

✓ Directorios verificados: 1
Imports OK


In [4]:
# ── 0.3 Configuración visual global ─────────────────────────────────────────
COLOR_PRINCIPAL  = "#3182ce"   # azul proyecto
COLOR_ABANDONO   = "#e53e3e"   # rojo  → abandona (clase 1)
COLOR_CONTINUA   = "#3182ce"   # azul  → no abandona (clase 0)
PALETTE_BINARIA  = {0: COLOR_CONTINUA, 1: COLOR_ABANDONO}
PALETTE_LABEL    = {"No abandona": COLOR_CONTINUA, "Abandona": COLOR_ABANDONO}
PALETTE_PLOTNINE = [COLOR_CONTINUA, COLOR_ABANDONO]

sns.set_theme(style="whitegrid", palette=[COLOR_CONTINUA, COLOR_ABANDONO])
plt.rcParams.update({"figure.dpi": 120, "axes.titlesize": 13, "axes.labelsize": 11})

# Etiquetas legibles para el target
LABEL_TARGET = {0: "No abandona", 1: "Abandona"}

print("Configuración visual OK")

# Helpers para embeber gráficos matplotlib/plotnine en HTML como base64
def fig_a_base64(fig, dpi=150, bbox="tight"):
    """Convierte figura Matplotlib a string base64 para embeber en HTML."""
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=dpi, bbox_inches=bbox,
                facecolor="white", edgecolor="none")
    plt.close(fig)
    buf.seek(0)
    return base64.b64encode(buf.read()).decode("utf-8")

def img_html(b64, ancho="100%"):
    """Genera tag <img> desde base64."""
    return (f'<img src="data:image/png;base64,{b64}" '
            f'style="width:{ancho};max-width:100%;display:block;margin:0 auto;">')

info_entorno()


Configuración visual OK
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓ ====

In [5]:
# ── 0.4 Carga del dataset de producción ─────────────────────────────────────
# Fase 4 lee siempre de 04_eda — texto legible, coherente con M01-M04
RUTA_DATOS = ROOT / "data" / "04_eda" / "df_eda_final.parquet"
assert RUTA_DATOS.exists(), f"No se encuentra el dataset en: {RUTA_DATOS}"

df = pd.read_parquet(RUTA_DATOS)

# Etiqueta legible en columna auxiliar (no modifica el dataset)
df["abandono_label"] = df["abandono"].map(LABEL_TARGET)

print(f"Dataset cargado: {formato_numero_es(df.shape[0])} filas × {df.shape[1]} columnas")
print(f"Tasa de abandono global: {df['abandono'].mean():.1%}")

Dataset cargado: 33.621 filas × 27 columnas
Tasa de abandono global: 29.2%


In [6]:
# ── 0.5 Clasificación de variables ──────────────────────────────────────────
TARGET = "abandono"
EXCLUIR = {TARGET, "abandono_label", "titulacion"}  # titulacion: alta cardinalidad, no para bivariante

# ── Clasificación dinámica — sin listas hardcodeadas ─────────────────────────
# df_eda_final tiene: str = categórica con texto, number = numérica
# titulacion excluida del bivariante (40 categorías — análisis separado en M02)
VARS_CAT = [
    c for c in df.columns
    if c not in EXCLUIR and df[c].dtype == 'object'
]
VARS_NUM = [
    c for c in df.columns
    if c not in EXCLUIR and df[c].dtype != 'object'
]

print(f"Variables numéricas ({len(VARS_NUM)}): {VARS_NUM}")
print(f"Variables categóricas ({len(VARS_CAT)}): {VARS_CAT}")
print(f"Excluidas: {EXCLUIR}")


Variables numéricas (16): ['anios_gap', 'anios_sin_beca', 'cred_repetidos', 'cred_superados_anio_1er', 'edad_entrada', 'indicador_interrupcion', 'max_pagos', 'n_anios_beca', 'n_anios_sin_notas', 'n_anios_trabajando', 'nota_1er_anio', 'nota_acceso', 'nota_selectividad', 'orden_preferencia', 'tasa_abandono_titulacion', 'tasa_repeticion']
Variables categóricas (8): ['cupo', 'pais_nombre', 'provincia', 'rama', 'sexo', 'situacion_laboral', 'universidad_origen', 'via_acceso']
Excluidas: {'abandono_label', 'abandono', 'titulacion'}


---
## 1 · Variables numéricas vs `abandono`
### 1a · Boxplots comparativos (Seaborn)

In [7]:
# ── 1a Boxplots: todas las numéricas en grid ─────────────────────────────────
n_vars = len(VARS_NUM)
n_cols = 3
n_rows = int(np.ceil(n_vars / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 4))
axes = axes.flatten()

for i, var in enumerate(VARS_NUM):
    ax = axes[i]
    # Boxplot con stripplot sobreimpuesto (muestra dispersión real)
    sns.boxplot(
        data=df, x="abandono_label", y=var,
        palette=PALETTE_LABEL, width=0.5,
        flierprops=dict(marker="o", markersize=2, alpha=0.3),
        ax=ax
    )
    # Añadir mediana como texto
    for j, clase in enumerate([0, 1]):
        mediana = df.loc[df[TARGET] == clase, var].median()
        ax.text(
            j, mediana, f" {mediana:.1f}",
            va="center", ha="left", fontsize=8, color="black", fontweight="bold"
        )
    ax.set_title(var, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel("")

# Ocultar ejes sobrantes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Variables numéricas: distribución por clase de abandono", fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()
# Capturar como base64 para el HTML
graf_boxplots = img_html(fig_a_base64(fig))


### 1b · Densidades KDE superpuestas por clase (Plotnine)

In [8]:
# ── 1b KDE en formato long para Plotnine ────────────────────────────────────
df_long_num = df[VARS_NUM + [TARGET, "abandono_label"]].melt(
    id_vars=[TARGET, "abandono_label"],
    value_vars=VARS_NUM,
    var_name="variable",
    value_name="valor"
)

p_kde = (
    ggplot(df_long_num, aes(x="valor", fill="abandono_label", color="abandono_label"))
    + geom_density(alpha=0.4, size=0.8)
    + facet_wrap("~ variable", scales="free", ncol=3)
    + scale_fill_manual(values=PALETTE_PLOTNINE)
    + scale_color_manual(values=PALETTE_PLOTNINE)
    + labs(
        title="Distribuciones KDE por clase de abandono",
        x="Valor", y="Densidad",
        fill="Clase", color="Clase"
    )
    + theme_minimal()
    + theme(
        figure_size=(15, n_rows * 3.5),
        plot_title=element_text(size=14, weight="bold"),
        strip_text=element_text(size=9, weight="bold"),
        legend_position="top"
    )
)

print(p_kde)
# Capturar plotnine como base64
buf_kde = io.BytesIO()
p_kde.save(buf_kde, format="png", dpi=150, verbose=False)
buf_kde.seek(0)
graf_kde = img_html(base64.b64encode(buf_kde.read()).decode("utf-8"))


<ggplot: (1500 x 2100)>


### 1c · Tests estadísticos: Mann-Whitney U + Cohen's *d* + IC 95 % (Pingouin + Scipy)

In [9]:
# ── 1c Función de análisis bivariante numérico ───────────────────────────────
def analizar_numerica_vs_target(df: pd.DataFrame, var: str, target: str) -> dict:
    """
    Calcula estadísticos bivariantes para una variable numérica vs target binario.
    Retorna: medianas, Mann-Whitney U (p-valor, r effect size), Cohen d + IC95% bootstrap.
    """
    g0 = df.loc[df[target] == 0, var].dropna()
    g1 = df.loc[df[target] == 1, var].dropna()

    # Mann-Whitney U
    U_stat, p_mw = stats.mannwhitneyu(g0, g1, alternative="two-sided")
    n0, n1 = len(g0), len(g1)
    # Effect size r = Z / sqrt(N)  (rank-biserial correlation)
    # Cuando p≈0, usar aproximación por U directamente (rank-biserial)
    r_effect = 1 - (2 * U_stat) / (n0 * n1)

    # Cohen's d con IC95% bootstrap via pingouin
    try:
        pg_result = pg.mwu(g1, g0, alternative="two-sided")
        # pingouin mwu ya da CLES (Common Language Effect Size)
        cles = pg_result["CLES"].values[0]
    except Exception:
        cles = np.nan

    # Cohen's d clásico (diferencia de medias / pooled std)
    pooled_std = np.sqrt(((n0 - 1) * g0.std()**2 + (n1 - 1) * g1.std()**2) / (n0 + n1 - 2))
    cohen_d = (g1.mean() - g0.mean()) / pooled_std if pooled_std > 0 else np.nan

    # IC 95% para Cohen's d via bootstrap
    def cohen_d_bootstrap(g0, g1, n_boot=500, seed=42):
        rng = np.random.default_rng(seed)
        ds = []
        for _ in range(n_boot):
            s0 = rng.choice(g0, size=min(len(g0), 2000), replace=True)
            s1 = rng.choice(g1, size=min(len(g1), 2000), replace=True)
            ps = np.sqrt(((len(s0)-1)*s0.std()**2 + (len(s1)-1)*s1.std()**2) / (len(s0)+len(s1)-2))
            ds.append((s1.mean() - s0.mean()) / ps if ps > 0 else np.nan)
        ds = np.array(ds)
        return np.nanpercentile(ds, 2.5), np.nanpercentile(ds, 97.5)

    ci_low, ci_high = cohen_d_bootstrap(g0.values, g1.values)

    # Interpretación automática de Cohen's d
    abs_d = abs(cohen_d) if not np.isnan(cohen_d) else 0
    if abs_d < 0.2:       magnitud = "trivial"
    elif abs_d < 0.5:     magnitud = "pequeño"
    elif abs_d < 0.8:     magnitud = "mediano"
    else:                 magnitud = "grande"

    return {
        "variable":      var,
        "mediana_no_ab": round(g0.median(), 3),
        "mediana_ab":    round(g1.median(), 3),
        "media_no_ab":   round(g0.mean(), 3),
        "media_ab":      round(g1.mean(), 3),
        "n_no_ab":       n0,
        "n_ab":          n1,
        "U_stat":        round(U_stat, 0),
        "p_mannwhitney": round(p_mw, 6),
        "r_effect":      round(r_effect, 4),
        "cohen_d":       round(cohen_d, 4) if not np.isnan(cohen_d) else np.nan,
        "cohen_d_ci95_low":  round(ci_low, 4),
        "cohen_d_ci95_high": round(ci_high, 4),
        "CLES":          round(cles, 4) if not np.isnan(cles) else np.nan,
        "magnitud":      magnitud,
        "significativo": p_mw < 0.05
    }


# Ejecutar para todas las numéricas
resultados_num = pd.DataFrame([
    analizar_numerica_vs_target(df, var, TARGET) for var in VARS_NUM
])
resultados_num = resultados_num.sort_values("cohen_d", key=abs, ascending=False)

print(f"Análisis completado para {len(VARS_NUM)} variables numéricas")
resultados_num

Análisis completado para 16 variables numéricas


,variable,mediana_no_ab,mediana_ab,media_no_ab,media_ab,n_no_ab,n_ab,U_stat,p_mannwhitney,r_effect,cohen_d,cohen_d_ci95_low,cohen_d_ci95_high,CLES,magnitud,significativo
7,n_anios_beca,2.00,1.000,2.096,0.850,23788,9833,169238188.0,0.000000,-0.4471,-0.8454,-0.9815,-0.8575,0.2765,grande,True
9,n_anios_trabajando,2.00,0.000,2.053,0.731,23788,9833,166248920.0,0.000000,-0.4215,-0.8135,-0.9572,-0.8192,0.2893,grande,True
10,nota_1er_anio,6.96,6.230,7.002,6.347,23391,7877,132520158.0,0.000000,-0.4385,-0.7805,-0.8571,-0.7258,0.2808,mediano,True
14,tasa_abandono_titulacion,0.23,0.382,0.274,0.365,23788,9833,72611777.0,0.000000,0.3791,0.6996,0.6420,0.7739,0.6896,mediano,True
11,nota_acceso,8.60,7.340,8.727,7.520,22103,8951,137567448.0,0.000000,-0.3907,-0.6926,-0.7849,-0.6524,0.3047,mediano,True
3,cred_superados_anio_1er,60.00,30.000,61.815,36.591,23788,9833,175569568.0,0.000000,-0.5012,-0.6239,-0.7054,-0.5782,0.2494,mediano,True
8,n_anios_sin_notas,0.00,0.000,0.151,0.440,23788,9833,88153329.0,0.000000,0.2463,0.5449,0.4401,0.5703,0.6231,mediano,True
12,nota_selectividad,6.75,6.180,6.897,6.383,21775,8728,123337821.0,0.000000,-0.2979,-0.5096,-0.5891,-0.4564,0.3510,mediano,True
2,cred_repetidos,0.00,0.000,18.689,6.695,23788,9833,158782274.0,0.000000,-0.3577,-0.3778,-0.4630,-0.3357,0.3212,pequeño,True
15,tasa_repeticion,0.00,0.000,7.752,2.787,23788,9833,158767002.0,0.000000,-0.3575,-0.3764,-0.4613,-0.3342,0.3212,pequeño,True


In [10]:
# ── 1d Tabla HTML estilizada de resultados numéricos ────────────────────────
def colorear_magnitud(val):
    colores = {
        "grande":  "background-color: #fed7d7; font-weight: bold",
        "mediano": "background-color: #fefcbf",
        "pequeño": "background-color: #e6fffa",
        "trivial": ""
    }
    return colores.get(val, "")

def colorear_p(val):
    return "color: #e53e3e; font-weight: bold" if val < 0.05 else "color: gray"

cols_mostrar = [
    "variable", "mediana_no_ab", "mediana_ab",
    "p_mannwhitney", "r_effect", "cohen_d",
    "cohen_d_ci95_low", "cohen_d_ci95_high", "magnitud"
]

styled_num = (
    resultados_num[cols_mostrar]
    .style
    .applymap(colorear_magnitud, subset=["magnitud"])
    .applymap(colorear_p, subset=["p_mannwhitney"])
    .set_caption("📊 Tests Mann-Whitney + Cohen's d con IC 95 % (bootstrap) — Variables numéricas")
    .format({
        "p_mannwhitney": "{:.4f}",
        "r_effect":      "{:.4f}",
        "cohen_d":       "{:+.4f}",
        "cohen_d_ci95_low":  "{:+.4f}",
        "cohen_d_ci95_high": "{:+.4f}",
    })
)

display(styled_num)

# Exportar CSV
RUTA_CSV_NUM = ROOT / "data" / "03_features" / "resultados_bivariante_numericas.csv"
RUTA_CSV_NUM.parent.mkdir(parents=True, exist_ok=True)
resultados_num.to_csv(RUTA_CSV_NUM, index=False, sep=";", decimal=",")
print(f"CSV guardado: {RUTA_CSV_NUM}")

,variable,mediana_no_ab,mediana_ab,p_mannwhitney,r_effect,cohen_d,cohen_d_ci95_low,cohen_d_ci95_high,magnitud
7,n_anios_beca,2.000000,1.000000,0.0000,-0.4471,-0.8454,-0.9815,-0.8575,grande
9,n_anios_trabajando,2.000000,0.000000,0.0000,-0.4215,-0.8135,-0.9572,-0.8192,grande
10,nota_1er_anio,6.960000,6.230000,0.0000,-0.4385,-0.7805,-0.8571,-0.7258,mediano
14,tasa_abandono_titulacion,0.230000,0.382000,0.0000,0.3791,+0.6996,+0.6420,+0.7739,mediano
11,nota_acceso,8.600000,7.340000,0.0000,-0.3907,-0.6926,-0.7849,-0.6524,mediano
3,cred_superados_anio_1er,60.000000,30.000000,0.0000,-0.5012,-0.6239,-0.7054,-0.5782,mediano
8,n_anios_sin_notas,0.000000,0.000000,0.0000,0.2463,+0.5449,+0.4401,+0.5703,mediano
12,nota_selectividad,6.750000,6.180000,0.0000,-0.2979,-0.5096,-0.5891,-0.4564,mediano
2,cred_repetidos,0.000000,0.000000,0.0000,-0.3577,-0.3778,-0.4630,-0.3357,pequeño
15,tasa_repeticion,0.000000,0.000000,0.0000,-0.3575,-0.3764,-0.4613,-0.3342,pequeño


CSV guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features\resultados_bivariante_numericas.csv


---
## 2 · Variables categóricas vs `abandono`
### 2a · Tasas de abandono por categoría (Plotly barras apiladas normalizadas)

In [11]:
# ── 2a Barras apiladas 100% por variable categórica (Plotly) ─────────────────
def plot_tasa_abandono_cat(df: pd.DataFrame, var: str, target: str) -> go.Figure:
    """Barra 100% apilada mostrando tasa de abandono por categoría."""
    tabla = (
        df.groupby([var, target])
        .size()
        .reset_index(name="n")
    )
    tabla["total"] = tabla.groupby(var)["n"].transform("sum")
    tabla["pct"] = tabla["n"] / tabla["total"] * 100
    tabla["label"] = tabla[target].map(LABEL_TARGET)

    # Ordenar categorías por tasa de abandono descendente
    orden = (
        tabla[tabla[target] == 1]
        .sort_values("pct", ascending=False)[var]
        .tolist()
    )

    fig = go.Figure()
    for clase, color, nombre in [(0, COLOR_CONTINUA, "No abandona"), (1, COLOR_ABANDONO, "Abandona")]:
        sub = tabla[tabla[target] == clase]
        sub = sub.set_index(var).reindex(orden).reset_index()
        fig.add_trace(go.Bar(
            name=nombre,
            x=sub[var].astype(str),
            y=sub["pct"],
            marker_color=color,
            text=sub["pct"].apply(lambda x: f"{x:.1f}%"),
            textposition="inside",
            hovertemplate=f"{nombre}<br>%{{x}}: %{{y:.1f}}%<extra></extra>"
        ))

    fig.update_layout(
        barmode="stack",
        title=dict(text=f"<b>Tasa de abandono por {var}</b>", font=dict(size=14)),
        xaxis_title=var, yaxis_title="Porcentaje (%)",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        height=420,
        template="plotly_white"
    )
    return fig


# Mostrar todas las categóricas
for var in VARS_CAT:
    fig = plot_tasa_abandono_cat(df, var, TARGET)
    fig.show()

### 2b · Tests Chi² + Cramér's *V* (Scipy)

In [12]:
# ── 2b Función de análisis bivariante categórico ─────────────────────────────
def analizar_categorica_vs_target(df: pd.DataFrame, var: str, target: str) -> dict:
    """
    Chi² de Pearson + Cramér's V para variable categórica vs target binario.
    Incluye tabla de contingencia, df, p-valor e interpretación de V.
    """
    tabla_cont = pd.crosstab(df[var], df[target])
    chi2, p_chi2, dof, expected = stats.chi2_contingency(tabla_cont)

    n = tabla_cont.values.sum()
    k = min(tabla_cont.shape) - 1  # min(filas, cols) - 1
    cramer_v = np.sqrt(chi2 / (n * k)) if (n * k) > 0 else np.nan

    # Interpretación de Cramér's V
    cv = cramer_v if not np.isnan(cramer_v) else 0
    if cv < 0.1:    magnitud = "trivial"
    elif cv < 0.3:  magnitud = "pequeña"
    elif cv < 0.5:  magnitud = "moderada"
    else:           magnitud = "fuerte"

    # Tasa de abandono por categoría (rango)
    tasas = df.groupby(var)[target].mean()

    return {
        "variable":       var,
        "n_categorias":   tabla_cont.shape[0],
        "chi2":           round(chi2, 3),
        "df":             dof,
        "p_chi2":         round(p_chi2, 6),
        "cramer_v":       round(cramer_v, 4) if not np.isnan(cramer_v) else np.nan,
        "magnitud":       magnitud,
        "tasa_min":       round(tasas.min(), 4),
        "tasa_max":       round(tasas.max(), 4),
        "rango_tasas":    round(tasas.max() - tasas.min(), 4),
        "cat_mayor_ab":   tasas.idxmax(),
        "cat_menor_ab":   tasas.idxmin(),
        "significativo":  p_chi2 < 0.05
    }


# Ejecutar para todas las categóricas
resultados_cat = pd.DataFrame([
    analizar_categorica_vs_target(df, var, TARGET) for var in VARS_CAT
])
resultados_cat = resultados_cat.sort_values("cramer_v", ascending=False)

print(f"Análisis completado para {len(VARS_CAT)} variables categóricas")
resultados_cat

Análisis completado para 8 variables categóricas


,variable,n_categorias,chi2,df,p_chi2,cramer_v,magnitud,tasa_min,tasa_max,rango_tasas,cat_mayor_ab,cat_menor_ab,significativo
7,via_acceso,13,820.133,12,0.000000,0.1562,pequeña,0.2207,0.5769,0.3562,Pruebas acceso mayores 40 años,Traslado,True
4,sexo,2,756.017,1,0.000000,0.1500,pequeña,0.2309,0.3681,0.1372,Hombre,Mujer,True
3,rama,5,585.568,4,0.000000,0.1320,pequeña,0.1590,0.3736,0.2146,TE,SA,True
0,cupo,10,198.268,9,0.000000,0.0828,trivial,0.0000,0.6286,0.6286,Mayor 40años,Diversidad funcional,True
2,provincia,50,211.260,49,0.000000,0.0793,trivial,0.0000,1.0000,1.0000,Ceuta,Palència,True
1,pais_nombre,77,131.117,76,0.000089,0.0624,trivial,0.0000,1.0000,1.0000,Cisjordania,Albania,True
5,situacion_laboral,11,56.165,10,0.000000,0.0519,trivial,0.1533,0.4333,0.2800,Fuerzas Armadas,Trabajadores calificados en la agricultura y l...,True
6,universidad_origen,5,46.904,4,0.000000,0.0403,trivial,0.2213,0.3094,0.0881,UV,UA,True


In [13]:
# ── 2c Tabla HTML estilizada de resultados categóricos ──────────────────────
def colorear_cramer(val):
    try:
        v = float(val)
        if v >= 0.5:   return "background-color: #fed7d7; font-weight: bold"
        elif v >= 0.3: return "background-color: #fefcbf"
        elif v >= 0.1: return "background-color: #e6fffa"
    except: pass
    return ""

cols_cat_mostrar = [
    "variable", "n_categorias", "chi2", "df",
    "p_chi2", "cramer_v", "magnitud",
    "tasa_min", "tasa_max", "rango_tasas",
    "cat_mayor_ab"
]

styled_cat = (
    resultados_cat[cols_cat_mostrar]
    .style
    .applymap(colorear_cramer, subset=["cramer_v"])
    .applymap(colorear_p, subset=["p_chi2"])
    .set_caption("📊 Tests Chi² + Cramér's V — Variables categóricas")
    .format({
        "chi2":       "{:.1f}",
        "p_chi2":     "{:.4f}",
        "cramer_v":   "{:.4f}",
        "tasa_min":   "{:.1%}",
        "tasa_max":   "{:.1%}",
        "rango_tasas":"{:.1%}",
    })
)

display(styled_cat)

# Exportar CSV
RUTA_CSV_CAT = ROOT / "data" / "03_features" / "resultados_bivariante_categoricas.csv"
resultados_cat.to_csv(RUTA_CSV_CAT, index=False, sep=";", decimal=",")
print(f"CSV guardado: {RUTA_CSV_CAT}")

,variable,n_categorias,chi2,df,p_chi2,cramer_v,magnitud,tasa_min,tasa_max,rango_tasas,cat_mayor_ab
7,via_acceso,13,820.1,12,0.0000,0.1562,pequeña,22.1%,57.7%,35.6%,Pruebas acceso mayores 40 años
4,sexo,2,756.0,1,0.0000,0.1500,pequeña,23.1%,36.8%,13.7%,Hombre
3,rama,5,585.6,4,0.0000,0.1320,pequeña,15.9%,37.4%,21.5%,TE
0,cupo,10,198.3,9,0.0000,0.0828,trivial,0.0%,62.9%,62.9%,Mayor 40años
2,provincia,50,211.3,49,0.0000,0.0793,trivial,0.0%,100.0%,100.0%,Ceuta
1,pais_nombre,77,131.1,76,0.0001,0.0624,trivial,0.0%,100.0%,100.0%,Cisjordania
5,situacion_laboral,11,56.2,10,0.0000,0.0519,trivial,15.3%,43.3%,28.0%,Fuerzas Armadas
6,universidad_origen,5,46.9,4,0.0000,0.0403,trivial,22.1%,30.9%,8.8%,UV


CSV guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features\resultados_bivariante_categoricas.csv


---
## 3 · Ranking global de asociación con `abandono`

In [14]:
# ── 3 Ranking unificado: numéricas (|cohen_d|) + categóricas (cramer_v) ──────
# Normalizamos a una escala común [0,1] para poder ordenar juntas
ranking_num = resultados_num[["variable", "cohen_d"]].copy()
ranking_num["asociacion"] = ranking_num["cohen_d"].abs()
ranking_num["tipo"] = "Numérica"
ranking_num["metrica"] = "|Cohen d|"

ranking_cat = resultados_cat[["variable", "cramer_v"]].copy()
ranking_cat["asociacion"] = ranking_cat["cramer_v"]
ranking_cat["tipo"] = "Categórica"
ranking_cat["metrica"] = "Cramér V"

ranking = pd.concat([
    ranking_num[["variable", "asociacion", "tipo", "metrica"]],
    ranking_cat[["variable", "asociacion", "tipo", "metrica"]]
], ignore_index=True).sort_values("asociacion", ascending=True)

# Lollipop chart con Plotly
fig_rank = go.Figure()

colores_tipo = {"Numérica": COLOR_CONTINUA, "Categórica": "#805ad5"}

for _, row in ranking.iterrows():
    color = colores_tipo[row["tipo"]]
    # Línea horizontal
    fig_rank.add_shape(
        type="line",
        x0=0, x1=row["asociacion"],
        y0=row["variable"], y1=row["variable"],
        line=dict(color=color, width=2)
    )

for tipo, color in colores_tipo.items():
    sub = ranking[ranking["tipo"] == tipo]
    fig_rank.add_trace(go.Scatter(
        x=sub["asociacion"],
        y=sub["variable"],
        mode="markers",
        marker=dict(size=12, color=color, line=dict(width=1, color="white")),
        name=tipo,
        text=sub.apply(lambda r: f"{r['metrica']}: {r['asociacion']:.3f}", axis=1),
        hovertemplate="<b>%{y}</b><br>%{text}<extra></extra>"
    ))

fig_rank.update_layout(
    title=dict(
        text="<b>Ranking de asociación con abandono</b><br>"
             "<sup>Numéricas: |Cohen d| · Categóricas: Cramér V</sup>",
        font=dict(size=15)
    ),
    xaxis_title="Fuerza de asociación",
    yaxis_title="",
    height=max(400, len(ranking) * 35),
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02)
)

# Líneas de referencia para Cohen d
for umbral, etiqueta in [(0.2, "pequeño"), (0.5, "mediano"), (0.8, "grande")]:
    fig_rank.add_vline(
        x=umbral, line_dash="dot", line_color="gray", line_width=1,
        annotation_text=etiqueta, annotation_position="top"
    )

fig_rank.show()

---
## 4 · Deep-dive variables clave

In [15]:
# ── 4a Identificar top variables automáticamente ─────────────────────────────
TOP_N = 4  # Número de variables a analizar en profundidad

# Top variables por |Cohen d|:
top_num_vars = (
    resultados_num
    .assign(_abs=resultados_num["cohen_d"].abs())
    .nlargest(TOP_N, "_abs")["variable"]
    .tolist()
)
top_cat_vars = resultados_cat.nlargest(TOP_N, "cramer_v")["variable"].tolist()

print(f"Top {TOP_N} numéricas:   {top_num_vars}")
print(f"Top {TOP_N} categóricas: {top_cat_vars}")

Top 4 numéricas:   ['n_anios_beca', 'n_anios_trabajando', 'nota_1er_anio', 'tasa_abandono_titulacion']
Top 4 categóricas: ['via_acceso', 'sexo', 'rama', 'cupo']


In [16]:
# ── 4b Violin + boxplot combinado para top numéricas (Seaborn) ───────────────
fig, axes = plt.subplots(1, len(top_num_vars), figsize=(5 * len(top_num_vars), 6))
if len(top_num_vars) == 1:
    axes = [axes]

for ax, var in zip(axes, top_num_vars):
    # Violin
    sns.violinplot(
        data=df, x="abandono_label", y=var,
        palette=PALETTE_LABEL, inner=None,
        alpha=0.6, ax=ax
    )
    # Boxplot interior
    sns.boxplot(
        data=df, x="abandono_label", y=var,
        palette=PALETTE_LABEL,
        width=0.15, fliersize=0, linewidth=1.5,
        boxprops=dict(alpha=0.9), ax=ax
    )
    # Estadísticos clave
    stats_var = resultados_num[resultados_num["variable"] == var].iloc[0]
    ax.set_title(
        f"{var}\nd={stats_var['cohen_d']:+.3f} · p={stats_var['p_mannwhitney']:.4f}",
        fontweight="bold", fontsize=11
    )
    ax.set_xlabel("")

fig.suptitle(f"Top {TOP_N} variables numéricas más asociadas con abandono",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()
# Capturar como base64
graf_violin = img_html(fig_a_base64(fig))


In [17]:
# ── 4c Deep-dive categóricas: tasas con IC Wilson (Statsmodels) ───────────────
from statsmodels.stats.proportion import proportion_confint

def tabla_tasa_con_ic(df: pd.DataFrame, var: str, target: str) -> pd.DataFrame:
    """Calcula tasa de abandono por categoría + IC 95% Wilson."""
    grp = df.groupby(var)[target].agg(["sum", "count"]).reset_index()
    grp.columns = [var, "n_abandona", "n_total"]
    grp["tasa"] = grp["n_abandona"] / grp["n_total"]
    ic = grp.apply(
        lambda r: proportion_confint(r["n_abandona"], r["n_total"], method="wilson"),
        axis=1, result_type="expand"
    )
    grp["ic_low"]  = ic[0]
    grp["ic_high"] = ic[1]
    return grp.sort_values("tasa", ascending=False)


for var in top_cat_vars:
    tasa_df = tabla_tasa_con_ic(df, var, TARGET)

    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=tasa_df[var].astype(str),
        y=tasa_df["tasa"],
        error_y=dict(
            type="data",
            symmetric=False,
            array=(tasa_df["ic_high"] - tasa_df["tasa"]).values,
            arrayminus=(tasa_df["tasa"] - tasa_df["ic_low"]).values,
            visible=True, color="rgba(0,0,0,0.4)"
        ),
        marker_color=COLOR_ABANDONO,
        text=tasa_df["tasa"].apply(lambda x: f"{x:.1%}"),
        textposition="outside",
        hovertemplate="%{x}<br>Tasa abandono: %{y:.1%}<br>n=%{customdata}<extra></extra>",
        customdata=tasa_df["n_total"]
    ))

    cramer = resultados_cat.loc[resultados_cat["variable"] == var, "cramer_v"].values[0]
    fig.update_layout(
        title=dict(
            text=f"<b>Tasa de abandono por {var}</b> — Cramér V = {cramer:.4f}",
            font=dict(size=14)
        ),
        yaxis_tickformat=".0%",
        yaxis_title="Tasa de abandono (IC 95% Wilson)",
        xaxis_title=var,
        template="plotly_white",
        height=450
    )
    fig.show()

In [18]:
# ── 4d Análisis temporal ── OMITIDO ────────────────────────────────────────
# El dataset final no contiene variable de año académico como columna directa.
# Este análisis se realizó en f4_m02_target.ipynb sobre el dataset original.
print("Celda 4d omitida: sin variable de año académico en dataset_final_tfm.parquet")


Celda 4d omitida: sin variable de año académico en dataset_final_tfm.parquet


---
## 5 · Conclusiones para el tribunal

In [19]:
# ── 5 Generación de conclusiones automáticas basadas en los resultados ───────
def interpretar_cohen(d):
    """Devuelve interpretación textual de Cohen's d."""
    ad = abs(d)
    if ad < 0.2:   return "sin efecto práctico"
    elif ad < 0.5: return "efecto pequeño"
    elif ad < 0.8: return "efecto mediano"
    else:          return "efecto grande"

# Variables numéricas significativas
sig_num = resultados_num[resultados_num["significativo"] == True].copy()
sig_num = sig_num.assign(_abs=sig_num["cohen_d"].abs()).sort_values("_abs", ascending=False)

# Variables categóricas significativas
sig_cat = resultados_cat[resultados_cat["significativo"] == True].sort_values("cramer_v", ascending=False)

print("=" * 60)
print("RESUMEN EJECUTIVO — Análisis Bivariante vs Abandono")
print("=" * 60)
print(f"\nVariables numéricas analizadas:   {len(VARS_NUM)}")
print(f"  → Significativas (p<0.05):       {len(sig_num)}")
print(f"  → Con efecto mediano o grande:   {len(sig_num[sig_num['_abs'] >= 0.5])}")
print(f"\nVariables categóricas analizadas: {len(VARS_CAT)}")
print(f"  → Significativas (p<0.05):       {len(sig_cat)}")
print(f"  → Asociación moderada o fuerte:  {len(sig_cat[sig_cat['cramer_v'] >= 0.3])}")

print("\n── TOP variables numéricas por fuerza de asociación ──")
for _, row in sig_num.head(5).iterrows():
    dir_efecto = "mayor" if row["cohen_d"] > 0 else "menor"
    print(f"  {row['variable']:35s}  d={row['cohen_d']:+.3f}  ({interpretar_cohen(row['cohen_d'])})  "
          f"→ quienes abandonan tienen {dir_efecto} valor")

print("\n── TOP variables categóricas por Cramér V ──")
for _, row in sig_cat.head(5).iterrows():
    print(f"  {row['variable']:35s}  V={row['cramer_v']:.4f}  ({row['magnitud']})  "
          f"→ mayor abandono en: {row['cat_mayor_ab']}")

RESUMEN EJECUTIVO — Análisis Bivariante vs Abandono

Variables numéricas analizadas:   16
  → Significativas (p<0.05):       16
  → Con efecto mediano o grande:   8

Variables categóricas analizadas: 8
  → Significativas (p<0.05):       8
  → Asociación moderada o fuerte:  0

── TOP variables numéricas por fuerza de asociación ──
  n_anios_beca                         d=-0.845  (efecto grande)  → quienes abandonan tienen menor valor
  n_anios_trabajando                   d=-0.814  (efecto grande)  → quienes abandonan tienen menor valor
  nota_1er_anio                        d=-0.780  (efecto mediano)  → quienes abandonan tienen menor valor
  tasa_abandono_titulacion             d=+0.700  (efecto mediano)  → quienes abandonan tienen mayor valor
  nota_acceso                          d=-0.693  (efecto mediano)  → quienes abandonan tienen menor valor

── TOP variables categóricas por Cramér V ──
  via_acceso                           V=0.1562  (pequeña)  → mayor abandono en: Pruebas acces

In [20]:
# ── 5b Exportar HTML final ──────────────────────────────────────────────────
nav_fases, nav_modulos = generar_html_navegacion_completa(
    fase_activa="fase4", modulo_activo="m05"
)

# Tablas estadísticas
tabla_num_html = resultados_num[cols_mostrar].to_html(
    index=False, classes="tabla-resultados", float_format="{:.4f}".format
)
tabla_cat_html = resultados_cat[cols_cat_mostrar].to_html(
    index=False, classes="tabla-resultados", float_format="{:.4f}".format
)
tabla_rank_html = ranking.sort_values("asociacion", ascending=False).to_html(
    index=False, classes="tabla-resultados", float_format="{:.4f}".format
)

# Gráficos Plotly como HTML inline
graf_ranking_html = fig_rank.to_html(full_html=False, include_plotlyjs="cdn")

# KPIs
kpis_html = generar_kpis_html([
    {"valor": str(len(VARS_NUM)), "titulo": "Variables numéricas", "color": "#3182ce"},
    {"valor": str(len(VARS_CAT)), "titulo": "Variables categóricas", "color": "#805ad5"},
    {"valor": str(len(sig_num)), "titulo": "Numéricas sig. (p<0.05)", "color": "#38a169"},
    {"valor": str(len(sig_cat)), "titulo": "Categóricas sig. (p<0.05)", "color": "#38a169"},
    {"valor": sig_num.iloc[0]["variable"] if len(sig_num) > 0 else "—",
     "titulo": "Top numérica", "color": "#e53e3e"},
])

s_num = generar_seccion_html(
    "1 · Variables numéricas vs abandono",
    graf_boxplots + graf_kde
    + "<h4>Tests Mann-Whitney U + Cohen's d con IC 95% bootstrap</h4>"
    + tabla_num_html
)

s_cat = generar_seccion_html(
    "2 · Variables categóricas vs abandono",
    "<p>Test: Chi² de Pearson. Effect size: Cramér's V. IC 95% Wilson para tasas.</p>"
    + tabla_cat_html
)

s_ranking = generar_seccion_html(
    "3 · Ranking global de asociación",
    graf_ranking_html + tabla_rank_html
)

s_deepdive = generar_seccion_html(
    "4 · Deep-dive variables clave",
    graf_violin
)

s_concl = generar_seccion_html(
    "5 · Conclusiones",
    f"""<ul>
    <li>Variables numéricas significativas: <strong>{len(sig_num)}</strong> de {len(VARS_NUM)}</li>
    <li>Variables categóricas significativas: <strong>{len(sig_cat)}</strong> de {len(VARS_CAT)}</li>
    <li>Top numérica: <strong>{sig_num.iloc[0]["variable"] if len(sig_num)>0 else "N/A"}</strong>
        (d={sig_num.iloc[0]["cohen_d"]:+.3f} — {sig_num.iloc[0]["magnitud"]})</li>
    <li>Top categórica: <strong>{sig_cat.iloc[0]["variable"] if len(sig_cat)>0 else "N/A"}</strong>
        (V={sig_cat.iloc[0]["cramer_v"]:.4f} — {sig_cat.iloc[0]["magnitud"]})</li>
    </ul>"""
)

contenido = kpis_html + s_num + s_cat + s_ranking + s_deepdive + s_concl

html = render_base_html(
    titulo="M05: Bivariante vs Target",
    icono="📊",
    subtitulo="Fase 4: EDA Final | TFM Abandono Universitario",
    nav_fases=nav_fases,
    nav_modulos=nav_modulos,
    contenido=contenido,
    notebook_nombre="f4_m05_bivariante.ipynb",
    notebook_carpeta="fase4_eda",
)

ruta_html = RUTA_FASE4_HTML / "m05_bivariante.html"
guardar_html(html, ruta_html)
print(f"\n✅ HTML generado: {ruta_html}")


✅ HTML guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase4\m05_bivariante.html

✅ HTML generado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase4\m05_bivariante.html


In [21]:
# ── 5c Resumen final de archivos generados ───────────────────────────────────
print("\n" + "=" * 55)
print("ARCHIVOS GENERADOS POR f4_m05")
print("=" * 55)
archivos = [
    ("CSV",  RUTA_CSV_NUM),
    ("CSV",  RUTA_CSV_CAT),
    ("HTML", RUTA_HTML),
    ("IMG",  ROOT / "docs" / "html" / "fase4" / "img" / "f4_m05_boxplots_numericas.png"),
    ("IMG",  ROOT / "docs" / "html" / "fase4" / "img" / "f4_m05_kde_numericas.png"),
    ("IMG",  ROOT / "docs" / "html" / "fase4" / "img" / "f4_m05_violin_top_num.png"),
]
for tipo, ruta in archivos:
    existe = "✅" if ruta.exists() else "⚠️ NO creado"
    print(f"  [{tipo}] {existe}  {ruta.relative_to(ROOT)}")

print("\nM05 completado ✓  →  Siguiente: f4_m06_correlaciones.ipynb")


ARCHIVOS GENERADOS POR f4_m05
  [CSV] ✅  data\03_features\resultados_bivariante_numericas.csv
  [CSV] ✅  data\03_features\resultados_bivariante_categoricas.csv
  [HTML] ✅  docs\html
  [IMG] ⚠️ NO creado  docs\html\fase4\img\f4_m05_boxplots_numericas.png
  [IMG] ⚠️ NO creado  docs\html\fase4\img\f4_m05_kde_numericas.png
  [IMG] ⚠️ NO creado  docs\html\fase4\img\f4_m05_violin_top_num.png

M05 completado ✓  →  Siguiente: f4_m06_correlaciones.ipynb
